# RAV-Med: Step 6d, second natural benchmark (SciFact)
**Why.** HealthContradict has only 81 questions. SciFact (Wadden et al., EMNLP 2020) has expert-written scientific claims with biomedical abstracts labelled SUPPORT or CONTRADICT by annotators, so it is a naturally written, claim-given test of the relation ERS is meant to detect.

**Data.** Claims from the public train and dev files (the test split has no public labels). Passage conditions per claim: **SUPPORT** abstract, **CONTRADICT** abstract, **NEI_ONTOPIC** (cited abstract of a claim that has no evidence), and **IRRELEVANT** (a random abstract the claim does not cite). Abstracts with mixed labels for one claim are dropped and counted. Licence: CC BY-NC 2.0 according to the dataset card; check it before redistributing anything, and do not redistribute the abstracts.

**Endpoints (fixed in advance).** AUROC with SUPPORT as the positive class against CONTRADICT, IRRELEVANT, NEI_ONTOPIC and all three pooled; cluster bootstrap over claims. Scorers: similarity only, contradiction only, ERS. Two passage units: the whole abstract (as in PubMedQA) and the most claim-similar 80-word window (as in HealthContradict). Also the share of each condition labelled SAFE / UNSAFE at T = 0.2.

**Caution.** SciFact abstracts are long; a whole abstract exceeds the 256-token NLI limit, so the window unit is reported alongside. Report both.

In [ ]:
!pip install -q sentence-transformers transformers accelerate bitsandbytes scikit-learn scipy datasets

In [ ]:
import os, re, json, random, hashlib, subprocess, shutil, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED)
rng = np.random.default_rng(SEED)

DRY_RUN = False        # True = tiny fake models, no GPU: checks that every cell runs in seconds. Set False for real results.
DEVICE = "cuda"
EMBED_MODEL = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"   # as in the paper
NLI_MODEL   = "roberta-large-mnli"                                        # as in the paper
N_BOOT = 2000
OUT = "step6_outputs"; os.makedirs(OUT, exist_ok=True)

# ---- platform helpers: the same notebook runs on Google Colab and on Kaggle ----
KAGGLE_ROOT = "/kaggle"
ON_KAGGLE = os.path.isdir(KAGGLE_ROOT + "/working")
def _find_in_kaggle_input(name):
    for root, _, fs in os.walk(KAGGLE_ROOT + "/input"):
        if name in fs: return os.path.join(root, name)
    return None
def ensure_file(name):
    """Make `name` available in the working directory (Kaggle: from an attached dataset; Colab: upload dialog)."""
    if os.path.exists(name): return
    if ON_KAGGLE:
        p = _find_in_kaggle_input(name)
        if p is None: raise FileNotFoundError(f"{name} not found. In Kaggle: Add Input -> upload it as a dataset and attach it to this notebook.")
        shutil.copy(p, name); print("copied", name, "from", p); return
    try:
        from google.colab import files; print("Upload", name); files.upload()
    except ImportError:
        raise FileNotFoundError(name)
def load_cache(path):
    """Read a JSON cache; if the file is truncated (disconnect during a write), fall back to the .bak copy."""
    for p in (path, path + ".bak"):
        if os.path.exists(p):
            try: return json.load(open(p))
            except Exception as e: print("cache file", p, "unreadable:", e)
    return {}
def save_cache(cache, path):
    """Write atomically (temp file, then rename) and keep one backup, so a disconnect cannot corrupt the cache."""
    tmp = path + ".tmp"; json.dump(cache, open(tmp, "w"))
    if os.path.exists(path): shutil.copy(path, path + ".bak")
    os.replace(tmp, path)
def restore_cache(path):
    """Kaggle only: if a cache file from an earlier session is attached as input, copy it back."""
    if ON_KAGGLE and not os.path.exists(path):
        p = _find_in_kaggle_input(os.path.basename(path))
        if p: shutil.copy(p, path); print("restored cache from", p)

T_SAFE = 0.2

In [ ]:
import torch
def _h(s): return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16) / 0xffffffff

if DRY_RUN:
    def embed(texts, bs=64):
        out = []
        for t in texts:
            v = np.random.default_rng(int(_h(t) * 1e9)).normal(size=16); out.append(v / np.linalg.norm(v))
        return np.array(out)
    def nli3(prem, hyp, bs=32):
        rows = []
        for p, h in zip(prem, hyp):
            r = np.random.default_rng(int(_h(p[:200] + "||" + h) * 1e9)).random(3); rows.append(r / r.sum())
        return np.array(rows)          # columns: contradiction, neutral, entailment
else:
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
    print("SBERT max_seq_length (as in all earlier runs):", embedder.max_seq_length)
    nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
    nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE).eval()
    _lab = {l.lower(): i for i, l in nli.config.id2label.items()}
    _IDX = [_lab["contradiction"], _lab["neutral"], _lab["entailment"]]
    def embed(texts, bs=64):
        return embedder.encode(list(texts), batch_size=bs, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=len(texts) > 500)
    @torch.no_grad()
    def nli3(prem, hyp, bs=32):
        prem, hyp = list(prem), list(hyp)
        order = np.argsort([len(p) for p in prem]); out = np.zeros((len(prem), 3))
        for i in range(0, len(prem), bs):
            idx = order[i:i+bs]
            enc = nli_tok([prem[j] for j in idx], [hyp[j] for j in idx], truncation=True, padding=True, max_length=256, return_tensors="pt").to(DEVICE)
            out[idx] = torch.softmax(nli(**enc).logits.float(), -1)[:, _IDX].cpu().numpy()
        return out
def contra(prem, hyp, bs=32): return nli3(prem, hyp, bs)[:, 0]
def cos(a, b): return (embed(list(a)) * embed(list(b))).sum(1)

from sklearn.metrics import roc_auc_score
def auroc_ci(y, x, cluster, n_boot=N_BOOT, seed=SEED):
    """AUROC with a cluster bootstrap (resample whole clusters, e.g. queries or claims)."""
    y = np.asarray(y); x = np.asarray(x, float); cl = np.asarray(cluster)
    a = roc_auc_score(y, x); r = np.random.default_rng(seed); u = np.unique(cl); pos = {c: np.where(cl == c)[0] for c in u}; b = []
    for _ in range(n_boot):
        idx = np.concatenate([pos[c] for c in r.choice(u, len(u), replace=True)])
        if len(np.unique(y[idx])) == 2: b.append(roc_auc_score(y[idx], x[idx]))
    return a, np.percentile(b, 2.5), np.percentile(b, 97.5)

def win(a, b): return (np.asarray(a) > np.asarray(b)).astype(float) + 0.5 * (np.asarray(a) == np.asarray(b))
def pairwise_ci(w, topic, n_boot=N_BOOT, seed=SEED):
    """Pairwise accuracy (pooled over pairs) with a question-level cluster bootstrap, as in the HealthContradict step."""
    w = np.asarray(w, float); topic = np.asarray(topic); u = np.unique(topic); r = np.random.default_rng(seed)
    s = np.array([w[topic == t].sum() for t in u]); n = np.array([(topic == t).sum() for t in u]); bi = r.integers(0, len(u), size=(n_boot, len(u)))
    b = s[bi].sum(1) / n[bi].sum(1)
    return w.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)

In [ ]:
import tarfile, urllib.request
SF_DIR = "scifact_data"
def _download_scifact():
    urllib.request.urlretrieve("https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz", "scifact.tar.gz")
    with tarfile.open("scifact.tar.gz") as t: t.extractall(".")
    if os.path.isdir("data") and not os.path.isdir(SF_DIR): os.rename("data", SF_DIR)
def _fake_scifact():
    """Tiny made-up SciFact-style files, used only if DRY_RUN is True and the real download is impossible."""
    os.makedirs(SF_DIR, exist_ok=True); r0 = np.random.default_rng(0)
    cp = [dict(doc_id=i, title=f"Title {i}", abstract=[f"Sentence {k} on topic {i%7} " + " ".join(map(str, r0.integers(0, 99, 30))) for k in range(6)], structured=False) for i in range(300)]
    cl = [dict(id=c, claim=f"Claim {c} about topic {c%7}", evidence=({str(c): [dict(sentences=[0], label="SUPPORT")]} if c % 3 == 0 else {str(c): [dict(sentences=[1], label="CONTRADICT")]} if c % 3 == 1 else {}), cited_doc_ids=[c, (c + 1) % 300]) for c in range(120)]
    open(f"{SF_DIR}/corpus.jsonl", "w").write("\n".join(map(json.dumps, cp)))
    open(f"{SF_DIR}/claims_train.jsonl", "w").write("\n".join(map(json.dumps, cl[:90]))); open(f"{SF_DIR}/claims_dev.jsonl", "w").write("\n".join(map(json.dumps, cl[90:])))
if not os.path.exists(f"{SF_DIR}/corpus.jsonl"):
    try: _download_scifact()
    except Exception as e:
        if not DRY_RUN: raise
        print("Real SciFact download failed in dry run; using tiny fake data:", e); _fake_scifact()
rd = lambda f: [json.loads(l) for l in open(f"{SF_DIR}/{f}", encoding="utf-8")]
corpus = {int(d["doc_id"]): (d["title"] + ". " + " ".join(d["abstract"])).strip() for d in rd("corpus.jsonl")}
claims_sf = rd("claims_train.jsonl") + rd("claims_dev.jsonl")          # the test split has no public labels
rows, mixed = [], 0
for c in claims_sf:
    ev = c.get("evidence", {})
    for d, lst in ev.items():
        labs = {e["label"] for e in lst}
        if len(labs) == 1: rows.append((c["id"], c["claim"], int(d), "SUPPORT" if "SUPPORT" in labs else "CONTRADICT"))
        else: mixed += 1
    if not ev:
        for d in c.get("cited_doc_ids", []): rows.append((c["id"], c["claim"], int(d), "NEI_ONTOPIC"))
sf = pd.DataFrame(rows, columns=["claim_id", "claim", "doc_id", "cond"])
# irrelevant control: a random abstract the claim does not cite
allids = np.array(sorted(corpus)); r2 = np.random.default_rng(SEED); irr = []
cited = {c["id"]: set(map(int, c.get("cited_doc_ids", []))) | set(map(int, c.get("evidence", {}).keys())) for c in claims_sf}
for cid, g in sf[sf.cond.isin(["SUPPORT", "CONTRADICT"])].groupby("claim_id"):
    while True:
        d = int(r2.choice(allids))
        if d not in cited[cid]: break
    irr.append((cid, g.claim.iloc[0], d, "IRRELEVANT"))
sf = pd.concat([sf, pd.DataFrame(irr, columns=sf.columns)], ignore_index=True).drop_duplicates(["claim_id", "doc_id"]).reset_index(drop=True)
sf = sf[sf.doc_id.isin(corpus)].copy(); sf["passage"] = sf.doc_id.map(corpus)
print("SciFact items by condition:", sf.cond.value_counts().to_dict(), "| claims:", sf.claim_id.nunique(), "| docs with mixed labels dropped:", mixed)

def _wins(text):
    w = text.split(); ch = [" ".join(w[i:i+80]) for i in range(0, len(w), 80)]
    if len(ch) > 1 and len(ch[-1].split()) < 15: ch = ch[:-1]
    return ch[:120] if ch else [text]

def score_scifact(df, mode):
    """mode 'full': whole abstract as one passage (as in PubMedQA); 'win': most claim-similar 80-word window (as in HealthContradict)."""
    if mode == "full":
        P = df.passage.tolist(); H = df.claim.tolist()
        S = cos(P, H); C = contra(P, H)
    else:
        wins, owner = [], []
        for i, p in enumerate(df.passage.tolist()):
            for w in _wins(p): wins.append(w); owner.append(i)
        owner = np.array(owner); WE = embed(wins); uc = df.claim.unique().tolist(); CE = dict(zip(uc, embed(uc)))
        best = {}
        for k, (i, w) in enumerate(zip(owner, wins)):
            s = float(WE[k] @ CE[df.claim.iloc[i]])
            if i not in best or s > best[i][0]: best[i] = (s, w)
        S = np.array([best[i][0] for i in range(len(df))]); C = contra([best[i][1] for i in range(len(df))], df.claim.tolist())
    out = df.copy(); out["S"] = S; out["C"] = C; out["ERS"] = S - C
    return out

def scifact_tables(scored, scorers, tag):
    tasks = {"SUPPORT vs CONTRADICT": ["CONTRADICT"], "SUPPORT vs IRRELEVANT": ["IRRELEVANT"], "SUPPORT vs NEI_ONTOPIC": ["NEI_ONTOPIC"], "SUPPORT vs all others (pooled)": ["CONTRADICT", "IRRELEVANT", "NEI_ONTOPIC"]}
    rows = []
    for tn, negs in tasks.items():
        x = scored[scored.cond.isin(["SUPPORT"] + negs)]
        if x.cond.nunique() < 2: continue
        for sn, (col, sg) in scorers.items():
            a, lo, hi = auroc_ci((x.cond == "SUPPORT").astype(int), sg * x[col], x.claim_id)
            rows.append(dict(passage_unit=tag, task=tn, scorer=sn, n_pos=int((x.cond == "SUPPORT").sum()), n_neg=int((x.cond != "SUPPORT").sum()), AUROC=a, lo=lo, hi=hi))
    return pd.DataFrame(rows)

In [ ]:
SCS = {"Similarity only (S)": ("S", 1), "Contradiction only (-C)": ("C", -1), "ERS (S - C)": ("ERS", 1)}
res = []
for mode, tag in [("full", "full abstract"), ("win", "top-1 80-word window")]:
    s_ = score_scifact(sf, mode); s_.to_csv(f"{OUT}/s6d_scifact_scores_{mode}_notext.csv", index=False, columns=["claim_id", "doc_id", "cond", "S", "C", "ERS"])
    t_ = scifact_tables(s_, SCS, tag); res.append(t_)
    sh = s_.groupby("cond").agg(n=("ERS", "size"), mean_S=("S", "mean"), mean_C=("C", "mean"), mean_ERS=("ERS", "mean"),
                                SAFE=("ERS", lambda v: (v >= T_SAFE).mean()), UNSAFE=("ERS", lambda v: (v < T_SAFE - 0.2).mean()))
    print(f"\n=== {tag} ==="); print(sh.round(3)); sh.to_csv(f"{OUT}/s6d_scifact_condition_summary_{mode}.csv")
T = pd.concat(res); print(T.round(3).to_string(index=False)); T.to_csv(f"{OUT}/s6d_scifact_auroc.csv", index=False)

In [ ]:
import shutil
shutil.make_archive("rav_med_step6d_outputs", "zip", OUT)
try:
    from google.colab import files; files.download("rav_med_step6d_outputs.zip")
except ImportError: pass
print("Saved rav_med_step6d_outputs.zip")